In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import cKDTree
import torch
from torch_geometric.data import Data
import json
import warnings
warnings.filterwarnings('ignore')

In [3]:
feature_dir  = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/feature_tensors")
temporal_dir = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/temporal_features")
prior_dir    = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/frozen_prior_global")
graph_dir    = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/graphs")

In [4]:
PATCHES = {
    "Kanto_Japan":      dict(minlat=34.5,  maxlat=37.2,  minlon=138.5, maxlon=141.5),
    "Tohoku_Japan":     dict(minlat=37.5,  maxlat=40.5,  minlon=140.5, maxlon=143.5),
    "Central_Chile":    dict(minlat=-36.5, maxlat=-33.5, minlon=-72.5, maxlon=-69.5),
    "Central_Turkey":   dict(minlat=36.5,  maxlat=39.0,  minlon=35.5,  maxlon=39.0),
    "Central_Nepal":    dict(minlat=27.0,  maxlat=29.7,  minlon=83.5,  maxlon=86.5),
    "North_Island_NZ":  dict(minlat=-40.5, maxlat=-37.5, minlon=174.5, maxlon=178.0),
    "Sumatra":          dict(minlat=-5.5,  maxlat=-2.0,  minlon=100.5, maxlon=104.5),
    "Kutch_India":      dict(minlat=21.5,  maxlat=24.5,  minlon=68.5,  maxlon=72.0),
    "Sichuan_China":    dict(minlat=29.5,  maxlat=32.5,  minlon=102.0, maxlon=105.5),
    "W_Australia":      dict(minlat=-32.0, maxlat=-29.0, minlon=117.0, maxlon=120.5),
    "S_Norway":         dict(minlat=58.5,  maxlat=61.5,  minlon=5.0,   maxlon=9.0),
    "Ordos_China":      dict(minlat=37.0,  maxlat=40.0,  minlon=107.5, maxlon=111.0),
}

# Temporal split boundaries
TRAIN_END = "2018-12-31"
VAL_END   = "2021-12-31"
# Test: 2022-01-01 to 2024-12-31

# Graph construction parameters
K_SPATIAL      = 8      # k nearest geographic neighbours
CORR_THRESHOLD = 0.3    # Pearson r threshold for temporal edges
CORR_WINDOW    = "count_30d"  # feature index 0 — 30-day event count
CORR_FEAT_IDX  = 0

In [5]:
with open(feature_dir / "feature_names.json") as f:
    feature_names = json.load(f)

TEMPORAL_FEATURE_NAMES = [
    'count_30d', 'count_90d', 'count_180d', 'count_365d',
    'count_m4_180d', 'mean_mag_90d', 'max_mag_90d', 'max_mag_365d',
    'mean_iet_90d', 'cv_iet_90d',
    'time_since_m3', 'time_since_m4',
    'bvalue_180d', 'omori_K', 'omori_p',
    'neighbour_count_30d', 'neighbour_max_mag_90d',
]

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1 = np.radians(lat1), np.radians(lon1)
    lat2, lon2 = np.radians(lat2), np.radians(lon2)
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

print(f"Graph construction parameters:")
print(f"  Spatial edges:  k={K_SPATIAL} geographic nearest neighbours")
print(f"  Temporal edges: Pearson r > {CORR_THRESHOLD} on {CORR_WINDOW}")
print(f"  Train/val/test: 2000-2018 / 2019-2021 / 2022-2024")

Graph construction parameters:
  Spatial edges:  k=8 geographic nearest neighbours
  Temporal edges: Pearson r > 0.3 on count_30d
  Train/val/test: 2000-2018 / 2019-2021 / 2022-2024


In [6]:
graph_stats = []

for patch_name, bounds in PATCHES.items():
    print(f"\n{'='*55}")
    print(f"Building graph: {patch_name}")

    # ── Load data ─────────────────────────────────────────────────────
    feat_tensor    = np.load(feature_dir  / f"{patch_name}_features.npy")
    temp_tensor    = np.load(temporal_dir / f"{patch_name}_temporal.npy")
    prior_probs    = np.load(prior_dir    / f"{patch_name}_prior_probs.npy")
    lons           = np.load(feature_dir  / f"{patch_name}_lons.npy")
    lats           = np.load(feature_dir  / f"{patch_name}_lats.npy")

    n_lat, n_lon, n_static   = feat_tensor.shape
    _, _, n_steps, n_temp    = temp_tensor.shape
    n_cells                  = n_lat * n_lon

    # Valid cell mask
    valid_mask = ~np.any(
        np.isnan(feat_tensor.reshape(-1, n_static)), axis=1
    )
    valid_idx  = np.where(valid_mask)[0]
    n_valid    = len(valid_idx)

    # Cell coordinates
    grid_lats, grid_lons = np.meshgrid(lats, lons, indexing='ij')
    cell_lats = grid_lats.ravel()[valid_idx]
    cell_lons = grid_lons.ravel()[valid_idx]

    print(f"  Valid cells: {n_valid} / {n_cells}")

    # ── Spatial edges (k=8 geographic nearest neighbours) ─────────────
    # Use lat/lon directly for kNN — at 0.1° resolution geographic
    # distance and Euclidean distance in lat/lon are proportional
    coords_2d = np.column_stack([cell_lats, cell_lons])
    tree      = cKDTree(coords_2d)

    # Query k+1 neighbours (first neighbour is the point itself)
    k_query   = min(K_SPATIAL + 1, n_valid)
    dists, nbr_idx = tree.query(coords_2d, k=k_query)

    spatial_src, spatial_dst, spatial_weights = [], [], []
    for i in range(n_valid):
        for j_pos in range(1, k_query):  # skip self (position 0)
            j   = nbr_idx[i, j_pos]
            d_km = haversine_km(
                cell_lats[i], cell_lons[i],
                cell_lats[j], cell_lons[j]
            )
            spatial_src.append(i)
            spatial_dst.append(j)
            spatial_weights.append(1.0 / (d_km + 1e-6))  # inverse distance

    spatial_src     = np.array(spatial_src)
    spatial_dst     = np.array(spatial_dst)
    spatial_weights = np.array(spatial_weights)

    # Make undirected — add reverse edges
    spatial_src_full = np.concatenate([spatial_src, spatial_dst])
    spatial_dst_full = np.concatenate([spatial_dst, spatial_src])
    spatial_w_full   = np.concatenate([spatial_weights, spatial_weights])

    # Deduplicate
    spatial_edges = np.unique(
        np.column_stack([spatial_src_full, spatial_dst_full]), axis=0
    )
    print(f"  Spatial edges:  {len(spatial_edges):,} "
          f"(k={K_SPATIAL} neighbours, undirected)")

    # ── Temporal correlation edges ─────────────────────────────────────
    # Use training period only to avoid leakage
    # Time steps for training period (2000-2018 = first 228 months)
    n_train = 228  # Jan 2000 to Dec 2018

    # Extract count_30d series per valid cell
    # Shape: (n_valid, n_train)
    count_series = temp_tensor.reshape(
        n_cells, n_steps, n_temp
    )[valid_idx, :n_train, CORR_FEAT_IDX]

    # Replace NaN with 0 for correlation computation
    count_series_filled = np.nan_to_num(count_series, nan=0.0)

    # Only compute correlations for cells with non-trivial variance
    # (sparse cells have all-zero series — correlation is undefined)
    has_variance = count_series_filled.std(axis=1) > 0.01
    print(f"  Cells with variance in count series: "
          f"{has_variance.sum()} / {n_valid}")

    temporal_src, temporal_dst, temporal_weights = [], [], []

    if has_variance.sum() >= 2:
        active_idx = np.where(has_variance)[0]
        active_series = count_series_filled[active_idx]

        # Normalise for correlation
        mean = active_series.mean(axis=1, keepdims=True)
        std  = active_series.std(axis=1, keepdims=True) + 1e-8
        active_norm = (active_series - mean) / std

        # Batch correlation matrix
        # Shape: (n_active, n_active)
        corr_matrix = (active_norm @ active_norm.T) / n_train

        # Find pairs above threshold
        rows, cols = np.where(
            (corr_matrix > CORR_THRESHOLD) &
            (np.eye(len(active_idx)) == 0)
        )
        for r, c in zip(rows, cols):
            i_global = active_idx[r]
            j_global = active_idx[c]
            if i_global < j_global:  # avoid duplicates
                temporal_src.append(i_global)
                temporal_dst.append(j_global)
                temporal_weights.append(float(corr_matrix[r, c]))

    temporal_src     = np.array(temporal_src)
    temporal_dst     = np.array(temporal_dst)
    temporal_weights = np.array(temporal_weights)

    # Make undirected
    if len(temporal_src) > 0:
        t_src_full = np.concatenate([temporal_src, temporal_dst])
        t_dst_full = np.concatenate([temporal_dst, temporal_src])
        t_w_full   = np.concatenate([temporal_weights, temporal_weights])
        temporal_edges = np.column_stack([t_src_full, t_dst_full])
    else:
        temporal_edges = np.zeros((0, 2), dtype=int)
        t_w_full       = np.array([])

    print(f"  Temporal edges: {len(temporal_edges):,} "
          f"(r>{CORR_THRESHOLD}, undirected)")

    # ── Fill neighbour temporal features ──────────────────────────────
    # neighbour_count_30d  (feat idx 15): mean count_30d of k neighbours
    # neighbour_max_mag_90d (feat idx 16): max of max_mag_90d of neighbours

    # Rebuild full temporal tensor for valid cells
    temp_valid = temp_tensor.reshape(
        n_cells, n_steps, n_temp
    )[valid_idx]  # (n_valid, n_steps, n_temp)

    # Build neighbour lookup from spatial edges
    from collections import defaultdict
    nbr_lookup = defaultdict(list)
    for s, d in spatial_edges:
        nbr_lookup[s].append(d)
        nbr_lookup[d].append(s)

    print(f"  Filling neighbour features...")
    for i in range(n_valid):
        nbrs = nbr_lookup[i]
        if len(nbrs) == 0:
            continue
        nbr_count30  = temp_valid[nbrs, :, 0]   # count_30d
        nbr_maxmag90 = temp_valid[nbrs, :, 6]   # max_mag_90d
        temp_valid[i, :, 15] = np.nanmean(nbr_count30,  axis=0)
        temp_valid[i, :, 16] = np.nanmax( nbr_maxmag90, axis=0)

    # ── Build PyG Data object ─────────────────────────────────────────
    # Combine spatial and temporal edges
    all_src = np.concatenate([
        spatial_edges[:, 0],
        temporal_edges[:, 0] if len(temporal_edges) > 0 else []
    ]).astype(int)
    all_dst = np.concatenate([
        spatial_edges[:, 1],
        temporal_edges[:, 1] if len(temporal_edges) > 0 else []
    ]).astype(int)

    # Edge type: 0=spatial, 1=temporal
    edge_type = np.concatenate([
        np.zeros(len(spatial_edges), dtype=int),
        np.ones(len(temporal_edges), dtype=int)
    ])

    # Edge weights
    all_weights = np.concatenate([
        # Spatial: use inverse-distance weights (recompute from edges)
        np.ones(len(spatial_edges)),
        t_w_full if len(temporal_edges) > 0 else []
    ])

    edge_index = torch.tensor(
        np.column_stack([all_src, all_dst]).T,
        dtype=torch.long
    )
    edge_attr = torch.tensor(
        np.column_stack([all_weights, edge_type]),
        dtype=torch.float
    )

    # Node features: static features for valid cells
    static_valid = feat_tensor.reshape(
        -1, n_static
    )[valid_idx]  # (n_valid, 12)

    # Prior probabilities for valid cells
    prior_valid = prior_probs.reshape(
        -1, 6
    )[valid_idx]  # (n_valid, 6)

    # Temporal features: (n_valid, n_steps, 17)
    # Already filled with neighbour features above

    # Train/val/test masks on time axis
    # 2000-2018: steps 0-227 (228 steps)
    # 2019-2021: steps 228-263 (36 steps)
    # 2022-2024: steps 264-299 (36 steps)
    train_mask = torch.zeros(n_steps, dtype=torch.bool)
    val_mask   = torch.zeros(n_steps, dtype=torch.bool)
    test_mask  = torch.zeros(n_steps, dtype=torch.bool)
    train_mask[:228] = True
    val_mask[228:264] = True
    test_mask[264:]   = True

    # Cell coordinates tensor
    cell_coords = torch.tensor(
        np.column_stack([cell_lats, cell_lons]),
        dtype=torch.float
    )

    # Build PyG Data object
    data = Data(
        # Graph structure
        edge_index = edge_index,
        edge_attr  = edge_attr,

        # Node features
        x_static   = torch.tensor(static_valid,  dtype=torch.float),
        x_prior    = torch.tensor(prior_valid,    dtype=torch.float),
        x_temporal = torch.tensor(temp_valid,     dtype=torch.float),

        # Metadata
        cell_coords   = cell_coords,
        valid_idx     = torch.tensor(valid_idx,   dtype=torch.long),
        grid_shape    = torch.tensor([n_lat, n_lon], dtype=torch.long),
        n_valid       = n_valid,
        patch_name    = patch_name,

        # Time masks
        train_mask = train_mask,
        val_mask   = val_mask,
        test_mask  = test_mask,
    )

    # Save
    torch.save(data, graph_dir / f"{patch_name}_graph.pt")

    # Stats
    n_sp = len(spatial_edges)
    n_te = len(temporal_edges)
    print(f"  Graph saved:")
    print(f"    Nodes:          {n_valid}")
    print(f"    Spatial edges:  {n_sp:,}")
    print(f"    Temporal edges: {n_te:,}")
    print(f"    Total edges:    {n_sp + n_te:,}")
    print(f"    x_static:       {data.x_static.shape}")
    print(f"    x_prior:        {data.x_prior.shape}")
    print(f"    x_temporal:     {data.x_temporal.shape}")

    graph_stats.append({
        'patch':          patch_name,
        'n_nodes':        n_valid,
        'n_spatial_edges':  n_sp,
        'n_temporal_edges': n_te,
        'n_total_edges':    n_sp + n_te,
        'cells_with_var':   int(has_variance.sum()),
    })


Building graph: Kanto_Japan
  Valid cells: 596 / 810
  Spatial edges:  4,912 (k=8 neighbours, undirected)
  Cells with variance in count series: 596 / 596
  Temporal edges: 104,450 (r>0.3, undirected)
  Filling neighbour features...
  Graph saved:
    Nodes:          596
    Spatial edges:  4,912
    Temporal edges: 104,450
    Total edges:    109,362
    x_static:       torch.Size([596, 12])
    x_prior:        torch.Size([596, 6])
    x_temporal:     torch.Size([596, 300, 17])

Building graph: Tohoku_Japan
  Valid cells: 502 / 900
  Spatial edges:  4,152 (k=8 neighbours, undirected)
  Cells with variance in count series: 502 / 502
  Temporal edges: 116,278 (r>0.3, undirected)
  Filling neighbour features...
  Graph saved:
    Nodes:          502
    Spatial edges:  4,152
    Temporal edges: 116,278
    Total edges:    120,430
    x_static:       torch.Size([502, 12])
    x_prior:        torch.Size([502, 6])
    x_temporal:     torch.Size([502, 300, 17])

Building graph: Central_Chil

In [7]:
stats_df = pd.DataFrame(graph_stats)
print("\n" + "="*70)
print("GRAPH CONSTRUCTION SUMMARY")
print("="*70)
print(stats_df.to_string(index=False))
stats_df.to_csv(graph_dir / "graph_stats.csv", index=False)

# Save temporal feature names for reference
with open(graph_dir / "temporal_feature_names.json", "w") as f:
    json.dump(TEMPORAL_FEATURE_NAMES, f, indent=2)

print(f"\nAll graphs saved to data/graphs/")
print(f"Load with: data = torch.load('data/graphs/{{patch}}_graph.pt')")


GRAPH CONSTRUCTION SUMMARY
          patch  n_nodes  n_spatial_edges  n_temporal_edges  n_total_edges  cells_with_var
    Kanto_Japan      596             4912            104450         109362             596
   Tohoku_Japan      502             4152            116278         120430             502
  Central_Chile      865             7060            303004         310064             865
 Central_Turkey      875             7132             66248          73380             667
  Central_Nepal      810             6606            164020         170626             636
North_Island_NZ      985             8030            132464         140494             912
        Sumatra     1246            10144            186726         196870            1053
    Kutch_India     1015             8270             31068          39338             334
  Sichuan_China     1050             8542            291976         300518             812
    W_Australia     1050             8542             20326   

## Insights

### Overview

<p>Graph construction is the architectural bridge between static feature engineering and the temporal GNN. Each patch is represented as a heterogeneous graph where nodes correspond to 0.1° grid cells and edges encode two distinct types of spatial relationship: geographic proximity and temporal activity correlation. The graph structure is patch-specific — node count, spatial connectivity, and temporal edge density all vary across patches reflecting genuine differences in grid size, patch extent, and seismicity patterns. The GNN backbone operates on node embeddings and graph edges rather than fixed-size tensors, making it naturally size-agnostic and capable of handling this variability without architectural modification.</p>

### Node Representation

<p>Each valid grid cell (cells with no NaN in the static feature vector) becomes a node in the graph. Invalid cells — those with NaN features due to ocean coverage — are excluded from the graph entirely rather than imputed, ensuring that the model never receives artificial signals from ocean cells. Node count per patch ranges from 502 (Tohoku, high ocean fraction) to 1,246 (Sumatra), with most land-dominated patches having 800–1,050 nodes. Each node carries three feature sets: a 12-dimensional static feature vector from the rasterised geological layers, a 6-dimensional frozen prior probability vector from the GMM-PCA k=6 clustering, and a (n_timesteps, 17)-dimensional temporal feature matrix encoding the full seismic history at that cell. These three feature sets are kept structurally separate throughout the architecture — static and prior features feed the geological encoder, temporal features feed the backbone — preserving the frozen/adaptive decomposition at the data level as well as the model level.</p>

### Spatial Edges

<p>Spatial edges connect each node to its k=8 geographically nearest neighbours, computed using a cKDTree query on cell centroid coordinates. The resulting undirected graph has approximately 8.2 edges per node across all patches, with minor variation from boundary effects at patch edges where cells have fewer than 8 valid neighbours. Edge weights are set to the inverse geographic distance in kilometres, so closer neighbours exert stronger influence during message passing. Spatial edges represent the physical intuition that nearby cells share geological structure and stress state — a rupture on a fault segment directly affects the stress field of adjacent cells more than distant ones, and this proximity relationship is regime-invariant and therefore appropriate to include in the frozen structural component of the graph.</p>

<p>Spatial edge counts scale linearly with node count as expected: Sumatra (10,144 edges, 1,246 nodes), Sichuan and Australia (8,542 edges, 1,050 nodes each), Kanto (4,912 edges, 596 nodes). The consistency of the edges-per-node ratio across patches with very different seismicity levels confirms that spatial connectivity is being constructed from geography alone, with no contamination from catalog information.</p>

### Temporal Correlation Edges

<p>Temporal edges connect cells whose 30-day event count time series exhibit Pearson correlation r > 0.3 during the training period (2000–2018). The correlation threshold of 0.3 was chosen to capture genuine spatial coherence of seismic activity while avoiding spurious connections between cells that happen to share a single coincidental event cluster. Correlations were computed exclusively on the training period to prevent future seismicity information from leaking into the graph structure — using the full 300-month time series would allow post-2018 aftershock sequences to influence which cells are connected, which would constitute temporal leakage into the evaluation period.</p>

<p>Temporal edge counts vary substantially across patches and are directly interpretable in terms of the seismic sequences experienced during the training period. Chile (303,004 temporal edges) and Sichuan (291,976) have by far the densest temporal graphs. In Chile, the 2010 Maule Mw 8.8 and 2015 Illapel Mw 8.3 sequences produced spatially coherent aftershock distributions that caused event count time series across many cells to spike simultaneously, creating high pairwise correlations across large portions of the patch. In Sichuan, the 2008 Wenchuan Mw 7.9 sequence produced a similar effect. These are not artefacts — the temporal graph is correctly capturing the spatial coherence of major seismic sequences as a structural property of the data. The backbone will learn to propagate information along these high-correlation edges, effectively encoding the spatial extent of influence of major events.</p>

<p>At the other extreme, Australia (20,326), Kutch (31,068), and Turkey (66,248) have sparse temporal graphs. Australia and Kutch have few events during training, so most cell pairs have near-zero event count variance and cannot form correlation edges. Turkey's 2023 Kahramanmaraş sequence falls entirely within the test period (post-2021) and therefore contributes no temporal correlation edges in the training graph — this is intentional and correct. The pre-2022 Turkish catalog is dominated by moderate background seismicity concentrated along the East Anatolian Fault corridor, producing correlations only among cells in close geographic proximity to the fault.</p>

### Cells with temporal variance

<p>A critical diagnostic is the fraction of cells with non-trivial variance in their 30-day event count series, which determines the fraction of cells capable of forming temporal correlation edges. Active patches show near-complete variance coverage: Kanto (596/596 = 100%), Chile (865/865 = 100%), Tohoku (502/502 = 100%), NZ (912/985 = 93%). Sparse patches show substantially lower variance fractions: Norway (289/960 = 30%), Australia (257/1050 = 24%), Kutch (334/1015 = 33%).</p>

<p>This variance pattern has a direct architectural implication. In sparse patches, the graph is dominated by spatial edges because temporal correlation edges are absent for most cells. The backbone's message passing in these patches relies primarily on geographic proximity rather than activity correlation. Since the frozen geological prior provides the regime context and spatial edges provide the local geological neighbourhood, the backbone can still make meaningful predictions in sparse patches — but it does so through geological conditioning and geographic interpolation rather than through learned temporal dynamics. This is precisely the behaviour the architecture is designed to produce: the frozen prior carries the weight of prediction when the catalog is uninformative, exactly the setting where cross-regional transfer is most needed.</p>

### Neighbor Feature Completion

<p>Two temporal features that depend on the graph structure — mean 30-day event count and maximum 90-day magnitude among geographic neighbours — were filled during graph construction using the spatial adjacency lists. These features capture the spatial propagation of seismic activity: elevated counts or large events in neighbouring cells provide evidence of regional stress changes that may not yet be reflected in the focal cell's own catalog. Filling these features during graph construction rather than as a preprocessing step ensures consistency between the feature values and the graph structure actually used during training.</p>

### Train/Val/Test Temporal Split

<p>A strict chronological split was applied: training (January 2000 – December 2018, 228 time steps), validation (January 2019 – December 2021, 36 time steps), test (January 2022 – December 2024, 36 time steps). This split is embedded in the graph object as boolean time masks and enforced throughout training and evaluation. The test period was chosen to include the February 2023 Turkey Kahramanmaraş sequence, the 2022 Fukushima offshore earthquake (Mw 7.3), and several other significant events that provide natural retrospective validation opportunities. No shuffling of time steps is ever applied — temporal ordering is preserved throughout.</p>

### Design Decisions

<p>The correlation threshold of r > 0.3 and the spatial k=8 neighbourhood size are hyperparameters subject to ablation (Ablation D — graph connectivity). The choice of 30-day event count as the correlation series rather than longer windows reflects a deliberate preference for capturing short-term seismic clustering rather than long-term background rate similarity — two cells in the same tectonic region will always have similar long-term rates, but only genuinely coupled cells will show synchronised short-term fluctuations. Using training-period correlations only for edge construction is a strict leakage prevention measure that will be explicitly stated in the Methods section and verified by the leakage audit described in the evaluation framework.</p>
